# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [3]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # === EJEMPLO — sustituir por los valores reales de la imagen ===
        self.start = (4, 0)  # (fila, columna)

        self.walls = {
            (1, 1), (1, 2), (1, 3),
            (3, 4),
        }

        self.slippery_states = {
            (2, 2), (2, 3),
        }

        self.terminal_states = {
            (0, 5): 10.0,   # meta / objetivo
            (2, 5): -10.0,  # trampa
        }

        self.danger_states = {
            (3, 1): -3,
            (0, 3): -3,
        }
        # === fin de ejemplo ===

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if (r, c) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        """R(s): recompensa asociada a estar en el estado s."""
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def _move(self, state, action):
        """Aplica el movimiento; si choca con pared/borde, se queda en el mismo lugar."""
        r, c = state
        dr, dc = action
        next_state = (r + dr, c + dc)
        if self.is_valid_state(next_state):
            return next_state
        return state

    def get_transition_probs(self, state, action):
        """
        Devuelve [(next_state, probability), ...]

        - Si el estado es terminal, no hay transición (se queda ahí, prob=1).
        - Si el estado es resbaloso: 0.7 de ir en la dirección deseada,
          0.15 y 0.15 de desviarse a cada lado (perpendicular).
        - Si no es resbaloso: 1.0 de ir en la dirección deseada (determinista).
        - Cualquier movimiento que choque con pared/borde deja al agente
          en el mismo estado (self._move ya maneja esto).
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Direcciones perpendiculares a la acción (para deslizamiento lateral)
        perpendiculars = {
            (-1, 0): [(0, -1), (0, 1)],  # UP -> lateral es LEFT/RIGHT
            (1, 0):  [(0, -1), (0, 1)],  # DOWN -> lateral es LEFT/RIGHT
            (0, -1): [(-1, 0), (1, 0)],  # LEFT -> lateral es UP/DOWN
            (0, 1):  [(-1, 0), (1, 0)],  # RIGHT -> lateral es UP/DOWN
        }

        if state in self.slippery_states:
            main_prob = 0.7
            side_prob = 0.15
            outcomes = [(action, main_prob)]
            for side_action in perpendiculars[action]:
                outcomes.append((side_action, side_prob))
        else:
            outcomes = [(action, 1.0)]

        # Agrupar resultados que caen en el mismo next_state (por si chocan con pared)
        probs = {}
        for act, p in outcomes:
            next_state = self._move(state, act)
            probs[next_state] = probs.get(next_state, 0.0) + p

        return list(probs.items())


### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [4]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")

Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [7]:
def value_iteration(mdp, tol=1e-4, max_iter=1000, verbose=False):
    """
    Value Iteration:

        V_{k+1}(s) = R(s) + gamma * max_a sum_{s'} T(s,a,s') V_k(s')

    Devuelve:
        V       -> dict {state: value}
        history -> lista de deltas por iteración (para graficar convergencia)
    """
    V = {s: 0.0 for s in mdp.states()}
    history = []

    for it in range(max_iter):
        V_new = {}
        delta = 0.0

        for s in mdp.states():
            if mdp.is_terminal(s):
                # en estado terminal no hay max_a: el valor es solo R(s)
                V_new[s] = mdp.get_reward(s)
            else:
                best_q = float('-inf')
                for a in mdp.actions:
                    q = 0.0
                    for next_s, p in mdp.get_transition_probs(s, a):
                        q += p * V[next_s]
                    if q > best_q:
                        best_q = q

                V_new[s] = mdp.get_reward(s) + mdp.gamma * best_q

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        history.append(delta)

        if verbose:
            print(f"Iteración {it+1}: delta = {delta:.6f}")

        if delta < tol:
            if verbose:
                print(f"✓ Convergió en {it+1} iteraciones (delta < {tol})")
            break

    return V, history

In [10]:
def extract_policy(mdp, V):
    """
    Extrae la política greedy respecto a V:

        pi(s) = argmax_a sum_{s'} T(s,a,s') V(s')
    """
    policy = {}

    for s in mdp.states():
        if mdp.is_terminal(s):
            policy[s] = None
            continue

        best_action = None
        best_q = float('-inf')

        for a in mdp.actions:
            q = sum(p * V[next_s] for next_s, p in mdp.get_transition_probs(s, a))
            if q > best_q:
                best_q = q
                best_action = a

        policy[s] = best_action

    return policy

In [11]:
grid = WarehouseMDP()

V_star, history = value_iteration(grid, tol=1e-4, verbose=True)
pi_star = extract_policy(grid, V_star)

# Mostrar V* en forma de cuadrícula
for r in range(grid.height):
    row_vals = []
    for c in range(grid.width):
        s = (r, c)
        if s in grid.walls:
            row_vals.append("  WALL  ")
        else:
            row_vals.append(f"{V_star[s]:7.2f}")
    print(" ".join(row_vals))

Iteración 1: delta = 10.000000
Iteración 2: delta = 9.000000
Iteración 3: delta = 8.100000
Iteración 4: delta = 7.290000
Iteración 5: delta = 4.941000
Iteración 6: delta = 4.446900
Iteración 7: delta = 4.002210
Iteración 8: delta = 3.601989
Iteración 9: delta = 3.241790
Iteración 10: delta = 2.917611
Iteración 11: delta = 0.395140
Iteración 12: delta = 0.355626
Iteración 13: delta = 0.084231
Iteración 14: delta = 0.054580
Iteración 15: delta = 0.019894
Iteración 16: delta = 0.009008
Iteración 17: delta = 0.003455
Iteración 18: delta = 0.001547
Iteración 19: delta = 0.000616
Iteración 20: delta = 0.000271
Iteración 21: delta = 0.000112
Iteración 22: delta = 0.000048
✓ Convergió en 22 iteraciones (delta < 0.0001)
   0.35    1.50    2.78    4.20    8.00   10.00
  -0.68   WALL     WALL     WALL      6.20    8.00
  -1.45   -0.50    0.56    2.35    4.58  -10.00
  -2.30   -2.99    0.01    1.12   WALL     -2.70
  -2.70   -1.89   -0.99    0.01   -0.99   -1.89



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [8]:
import random

def initialize_random_policy(mdp):
    """
    Política inicial arbitraria (aleatoria) para estados no terminales.
    """
    policy = {}
    for s in mdp.states():
        if mdp.is_terminal(s):
            policy[s] = None
        else:
            policy[s] = random.choice(mdp.actions)
    return policy


def policy_evaluation(mdp, policy, tol=1e-4, max_iter=1000):
    """
    Policy Evaluation (exacta, iterativa):

        V_{k+1}^pi(s) = R(s) + gamma * sum_{s'} T(s, pi(s), s') V_k^pi(s')

    Devuelve V^pi en convergencia.
    """
    V = {s: 0.0 for s in mdp.states()}

    for _ in range(max_iter):
        V_new = {}
        delta = 0.0

        for s in mdp.states():
            if mdp.is_terminal(s):
                V_new[s] = mdp.get_reward(s)
            else:
                a = policy[s]
                q = sum(p * V[next_s] for next_s, p in mdp.get_transition_probs(s, a))
                V_new[s] = mdp.get_reward(s) + mdp.gamma * q

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new
        if delta < tol:
            break

    return V


def policy_improvement(mdp, V):
    """
    Policy Improvement:

        pi_new(s) = argmax_a sum_{s'} T(s,a,s') V(s')

    Devuelve la nueva política greedy respecto a V.
    """
    new_policy = {}

    for s in mdp.states():
        if mdp.is_terminal(s):
            new_policy[s] = None
            continue

        best_action = None
        best_q = float('-inf')

        for a in mdp.actions:
            q = sum(p * V[next_s] for next_s, p in mdp.get_transition_probs(s, a))
            if q > best_q:
                best_q = q
                best_action = a

        new_policy[s] = best_action

    return new_policy


def policy_iteration(mdp, tol=1e-4, max_iter=1000, verbose=False):
    """
    Policy Iteration completo:
        1. Evaluar la política actual -> V^pi
        2. Mejorar la política respecto a V^pi -> pi_new
        3. Repetir hasta que la política no cambie (convergencia)

    Devuelve:
        policy -> política óptima pi*
        V      -> V^pi* asociada
        n_iter -> número de iteraciones hasta converger
    """
    policy = initialize_random_policy(mdp)

    for it in range(max_iter):
        V = policy_evaluation(mdp, policy, tol=tol, max_iter=max_iter)
        new_policy = policy_improvement(mdp, V)

        if verbose:
            n_changes = sum(
                1 for s in mdp.states()
                if not mdp.is_terminal(s) and new_policy[s] != policy[s]
            )
            print(f"Iteración {it+1}: {n_changes} acciones cambiaron")

        if new_policy == policy:
            if verbose:
                print(f"✓ Política convergió en {it+1} iteraciones")
            policy = new_policy
            break

        policy = new_policy

    V = policy_evaluation(mdp, policy, tol=tol, max_iter=max_iter)
    return policy, V, it + 1

In [9]:
grid = WarehouseMDP()

pi_star, V_star, n_iter = policy_iteration(grid, verbose=True)

print(f"\nConvergió en {n_iter} iteraciones\n")

# Mostrar V^pi* en forma de cuadrícula
for r in range(grid.height):
    row_vals = []
    for c in range(grid.width):
        s = (r, c)
        if s in grid.walls:
            row_vals.append("  WALL  ")
        else:
            row_vals.append(f"{V_star[s]:7.2f}")
    print(" ".join(row_vals))

Iteración 1: 17 acciones cambiaron
Iteración 2: 5 acciones cambiaron
Iteración 3: 8 acciones cambiaron
Iteración 4: 6 acciones cambiaron
Iteración 5: 1 acciones cambiaron
Iteración 6: 0 acciones cambiaron
✓ Política convergió en 6 iteraciones

Convergió en 6 iteraciones

   0.35    1.50    2.78    4.20    8.00   10.00
  -0.68   WALL     WALL     WALL      6.20    8.00
  -1.45   -0.50    0.56    2.35    4.58  -10.00
  -2.30   -2.99    0.01    1.12   WALL     -2.70
  -2.70   -1.89   -0.99    0.01   -0.99   -1.89



## Parte 4 — Visualización y comparación


In [12]:
import random

ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


grid = WarehouseMDP()

# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")

=== VALUE ITERATION ===
Iteraciones: [10.0, 9.0, 8.1, 7.29, 4.941000000000001, 4.446900000000001, 4.002210000000001, 3.6019889999999997, 3.2417901, 2.9176110899999994, 0.3951398422433252, 0.3556258580189926, 0.08423107888625214, 0.05457973739895161, 0.019893996759187793, 0.009007754079499719, 0.0034551438596461814, 0.0015473308310003375, 0.0006164895014912908, 0.00027122677866753264, 0.00011151908955175927, 4.8009130697135305e-05]

Valores:
 +0.352 |  +1.502 |  +2.780 |  +4.200 |  +8.000 | +10.000
 -0.683 |   WALL   |   WALL   |   WALL   |  +6.200 |  +8.000
 -1.447 |  -0.496 |  +0.560 |  +2.354 |  +4.580 | -10.000
 -2.302 |  -2.994 |  +0.007 |  +1.119 |   WALL   |  -2.705
 -2.705 |  -1.894 |  -0.994 |  +0.007 |  -0.994 |  -1.894

Política:
 →  |  →  |  →  |  →  |  →  | +10
 ↑  |  #  |  #  |  #  |  ↑  |  ↑ 
 →  |  →  |  →  |  →  |  ↑  | -10
 ↑  |  →  |  →  |  ↑  |  #  |  ↓ 
 →  |  →  |  ↑  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: 6

Valores:
 +0.352 |  +1.502 |  +2.780 |  +

In [13]:
for s in grid.states():
    assert abs(V_vi[s] - V_pi[s]) < 1e-2, f"Discrepancia en {s}"
print("✓ Ambos algoritmos convergen al mismo V*")

✓ Ambos algoritmos convergen al mismo V*



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [14]:
## Experimento A
grid_a = WarehouseMDP()
grid_a.living_reward = -0.1

V_a, _ = value_iteration(grid_a)
pi_a = extract_policy(grid_a, V_a)

print("=== Experimento A (living_reward = -0.1) ===")
print_policy(grid_a, pi_a)
print("Acción desde START:", pi_a[grid_a.start])

=== Experimento A (living_reward = -0.1) ===
 →  |  →  |  →  |  →  |  →  | +10
 ↓  |  #  |  #  |  #  |  ↑  |  ↑ 
 →  |  →  |  →  |  →  |  ↑  | -10
 ↑  |  →  |  →  |  ↑  |  #  |  ↓ 
 →  |  →  |  ↑  |  ↑  |  ←  |  ← 
Acción desde START: (0, 1)


In [15]:
## Experimento B
def get_transition_probs_b(self, state, action):
    if self.is_terminal(state):
        return [(state, 1.0)]

    perpendiculars = {
        (-1, 0): [(0, -1), (0, 1)],
        (1, 0):  [(0, -1), (0, 1)],
        (0, -1): [(-1, 0), (1, 0)],
        (0, 1):  [(-1, 0), (1, 0)],
    }

    if state in self.slippery_states:
        main_prob = 0.40
        side_prob = 0.30   # (1 - 0.40) / 2
        outcomes = [(action, main_prob)]
        for side_action in perpendiculars[action]:
            outcomes.append((side_action, side_prob))
    else:
        outcomes = [(action, 1.0)]

    probs = {}
    for act, p in outcomes:
        next_state = self._move(state, act)
        probs[next_state] = probs.get(next_state, 0.0) + p

    return list(probs.items())


import types
grid_b = WarehouseMDP()
grid_b.get_transition_probs = types.MethodType(get_transition_probs_b, grid_b)

V_b, _ = value_iteration(grid_b)
pi_b = extract_policy(grid_b, V_b)

print("=== Experimento B (piso resbaloso 0.40) ===")
print_policy(grid_b, pi_b)

=== Experimento B (piso resbaloso 0.40) ===
 →  |  →  |  →  |  →  |  →  | +10
 ↑  |  #  |  #  |  #  |  ↑  |  ↑ 
 ↑  |  →  |  →  |  →  |  ↑  | -10
 ↑  |  →  |  →  |  ↑  |  #  |  ↓ 
 ↑  |  →  |  ↑  |  ↑  |  ←  |  ← 


In [16]:
## Experimento C
grid_c = WarehouseMDP()
grid_c.gamma = 0.99

V_c, _ = value_iteration(grid_c)
pi_c = extract_policy(grid_c, V_c)

print("=== Experimento C (gamma = 0.99) ===")
print_policy(grid_c, pi_c)
print("Acción desde START:", pi_c[grid_c.start])

=== Experimento C (gamma = 0.99) ===
 →  |  →  |  →  |  →  |  →  | +10
 ↑  |  #  |  #  |  #  |  ↑  |  ↑ 
 →  |  →  |  →  |  →  |  ↑  | -10
 ↑  |  →  |  →  |  ↑  |  #  |  ↓ 
 →  |  →  |  ↑  |  ↑  |  ←  |  ← 
Acción desde START: (0, 1)


In [18]:
## Experimento Bonus
def find_living_reward_threshold(base_grid, low=-2.0, high=0.0, tol=1e-3):
    """
    Busca (por bisección) el valor aproximado de living_reward donde
    la acción óptima desde start cambia.
    Asume que a 'low' se prefiere ir a la carga (+2) y a 'high' a la entrega (+10).
    """
    def action_at(lr):
        g = WarehouseMDP()
        g.living_reward = lr
        V, _ = value_iteration(g)
        pi = extract_policy(g, V)
        return pi[g.start]

    a_low = action_at(low)
    a_high = action_at(high)

    if a_low == a_high:
        print("No se detectó cambio de política en ese rango; amplía el intervalo.")
        return None

    while high - low > tol:
        mid = (low + high) / 2
        a_mid = action_at(mid)
        if a_mid == a_low:
            low = mid
        else:
            high = mid

    print(f"Umbral aproximado de living_reward: {(low+high)/2:.4f}")
    return (low + high) / 2

find_living_reward_threshold(WarehouseMDP)

Umbral aproximado de living_reward: -1.7134


-1.71337890625

**1. Desde `START`, ¿el robot busca la entrega +10 o prefiere la carga +2?**

Depende de la distancia efectiva a cada una, no del tamaño de la recompensa en sí. El agente no compara $+10$ contra $+2$ directamente: compara el **retorno descontado neto** de cada ruta,

$$V(\text{ruta}) \approx \gamma^{d}\, R - (\text{suma de costos de paso y penalizaciones en el camino})$$

Si la entrega está considerablemente más lejos que la carga, el descuento $\gamma^d$ y el costo acumulado de pasos pueden reducir su valor efectivo por debajo del de la carga cercana. Por eso la respuesta correcta no es "el robot prefiere la recompensa más grande", sino "el robot prefiere la ruta con mayor valor esperado descontado", y eso solo se puede determinar corriendo `value_iteration` sobre el grid real y observando `pi[start]`.

---

**2. ¿Por qué una recompensa menor podría ser óptima?**

Porque en un MDP con descuento y costo de vivir, lo que se maximiza es el **retorno acumulado**, no la recompensa terminal aislada. Una recompensa grande pero lejana sufre triple penalización:

- **Descuento temporal**: $\gamma^d$ decrece exponencialmente con la distancia $d$, así que entre más lejos esté, menos "vale" hoy.
- **Costo de oportunidad**: cada paso adicional cuesta `living_reward` (típicamente negativo), así que un camino más largo acumula más costo.
- **Riesgo en el camino**: más pasos significan más exposición a casillas de peligro o pisos resbaladizos que pueden desviar al agente y alargar aún más la ruta.

Si la suma de estas penalizaciones supera la diferencia entre las dos recompensas, la meta "menor pero cercana y segura" gana en valor esperado. Es el mismo principio que explica por qué, en la vida real, uno a veces prefiere una ganancia modesta e inmediata sobre una mayor pero incierta y distante.

---

**3. ¿En qué estados el piso resbaloso cambia la decisión?**

Conceptualmente, el piso resbaloso cambia la política en los estados donde:

- una **desviación lateral no intencional** (la probabilidad que no va a la dirección deseada) puede llevar al agente a una casilla de **peligro** o a una **trampa terminal**, aumentando el costo esperado de tomar esa ruta; o
- existe una **ruta alternativa determinista** (sin resbalones) cuya longitud extra es compensada por evitar el riesgo de la ruta corta pero incierta.

En otras palabras: el piso resbaloso importa cuando **la varianza del resultado tiene consecuencias asimétricas** — si deslizarse no cambia mucho el resultado (por ejemplo, las tres direcciones posibles llevan a casillas igualmente neutras), la política no cambia. Pero si una de las desviaciones posibles es catastrófica (cae en peligro o en una trampa), el valor esperado de esa acción baja lo suficiente como para que el agente prefiera un camino más largo pero seguro.

---

**4. ¿Qué papel cumple el costo por paso `-1`?**

Es el mecanismo que **traduce el tiempo en costo**, y cumple tres funciones:

- **Incentiva rutas cortas**: entre dos caminos que llegan al mismo destino, siempre se prefiere el de menor número de pasos, sin necesidad de penalizaciones explícitas en las celdas intermedias.
- **Crea el trade-off distancia-vs-recompensa**: es lo que permite que una recompensa cercana y menor le gane a una lejana y mayor (pregunta 1 y 2). Sin este costo (`living_reward = 0`), casi cualquier recompensa positiva lejana sería preferible, ya que "esperar" no tendría costo.
- **Actúa como una función de urgencia**: cuanto más negativo es `living_reward`, más "impaciente" se vuelve el agente, favoreciendo terminar rápido incluso a costa de una recompensa menor.

Es conceptualmente equivalente a una tasa de descuento adicional expresada en unidades absolutas en vez de multiplicativas.

---

**5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?**

Porque la dinámica del entorno **no es homogénea**: el piso tiene al menos dos regímenes físicos distintos —normal (determinista) y resbaloso (estocástico)— y la función de transición debe reflejar esa heterogeneidad.

Si se usara una única distribución de probabilidad fija para todo el espacio de estados, ocurriría uno de dos errores:

- **Subestimar el riesgo real**: tratar las casillas resbalosas como deterministas ignoraría la posibilidad de desviación, haciendo que el agente subestime el peligro de cruzarlas.
- **Sobreestimar la incertidumbre**: tratar las casillas normales como resbalosas introduciría ruido artificial donde el movimiento en realidad es predecible, distorsionando las decisiones óptimas también en esas zonas.

Por eso $T(s,a,s')$ debe estar **condicionada al estado de origen** ($s \in \text{slippery\_states}$ o no), no solo a la acción $a$: la transición depende de las propiedades físicas de la casilla desde la que se actúa, no únicamente de qué botón presiona el agente.